# Sentiment Analysis on Financial News
---
> **Bathaix Philippe-Emmanuel Yao**

This notebook implements a full NLP pipeline for three-class sentiment classification
(positive / neutral / negative) on financial news headlines, using the
[Financial PhraseBank](https://www.kaggle.com/datasets/ankurzing/sentiment-analysis-for-financial-news) dataset (4,846 sentences).

| Stage | Methods |
|---|---|
| **Lexicon-based** | VADER, TextBlob |
| **ML classifiers** | Logistic Regression, SVM, Random Forest (TF-IDF features) |
| **Evaluation** | Accuracy, F1 (macro & weighted), confusion matrix, class imbalance analysis |

**Improvements over original:**
- Google Drive dependency removed; loads `all-data.csv` locally
- Class imbalance explicitly handled: stratified splits + class-weight balancing
- Pipeline objects (`Pipeline` + `GridSearchCV`) for leakage-free cross-validation
- SMOTE oversampling option for the minority (negative) class
- Per-class F1 tracked and visualised, not just accuracy
- Model persistence with `joblib`; inference helper function included
- Full unit test suite with `pytest`-compatible assertions


## 1. Imports & Configuration

In [ ]:
import os
import logging
import warnings
import joblib
import unittest

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer

from textblob import TextBlob

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, f1_score)
from sklearn.utils.class_weight import compute_class_weight

warnings.filterwarnings('ignore')
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)s | %(message)s',
    handlers=[logging.StreamHandler(), logging.FileHandler('sentiment_analysis.log')]
)
log = logging.getLogger(__name__)

# Download VADER lexicon if not present
nltk.download('vader_lexicon', quiet=True)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': 0.3})

LABEL_ORDER = ['negative', 'neutral', 'positive']
LABEL_MAP   = {'negative': 0, 'neutral': 1, 'positive': 2}
LABEL_IMAP  = {v: k for k, v in LABEL_MAP.items()}


## 2. Data Loading & Preprocessing

In [ ]:
def load_data(file_path: str, encoding: str = 'ISO-8859-1') -> pd.DataFrame:
    """
    Load the Financial PhraseBank CSV.
    Expected format: two columns — sentiment label and news headline.
    No header row in the original file.
    """
    log.info(f"Loading data from: {file_path}")
    df = pd.read_csv(file_path, encoding=encoding, header=None, names=['sentiment', 'news'])
    log.info(f"Loaded {len(df):,} rows")
    return df


def preprocess_data(df: pd.DataFrame) -> pd.DataFrame:
    """
    Clean the dataset:
      1. Drop exact duplicates
      2. Drop rows with missing values
      3. Strip whitespace from text and labels
      4. Validate that all labels are in {negative, neutral, positive}
    """
    initial = len(df)
    df = df.drop_duplicates().dropna().copy()
    df['sentiment'] = df['sentiment'].str.strip().str.lower()
    df['news']      = df['news'].str.strip()

    valid_labels = set(LABEL_ORDER)
    invalid = df[~df['sentiment'].isin(valid_labels)]
    if len(invalid) > 0:
        log.warning(f"Dropping {len(invalid)} rows with unrecognised labels")
        df = df[df['sentiment'].isin(valid_labels)]

    log.info(f"After cleaning: {len(df):,} rows (removed {initial - len(df)})")
    return df.reset_index(drop=True)


def explore_data(df: pd.DataFrame) -> None:
    """Print class distribution and plot a bar chart."""
    counts = df['sentiment'].value_counts().reindex(LABEL_ORDER)
    total  = len(df)
    print(f"Dataset: {total:,} samples")
    print("\nClass distribution:")
    for label, cnt in counts.items():
        print(f"  {label:<12} {cnt:>5}  ({100*cnt/total:.1f}%)")

    fig, ax = plt.subplots(figsize=(7, 4))
    bars = ax.bar(counts.index, counts.values,
                  color=['#e74c3c', '#3498db', '#2ecc71'], edgecolor='white', linewidth=1.2)
    for bar, cnt in zip(bars, counts.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
                f'{cnt}\n({100*cnt/total:.1f}%)', ha='center', va='bottom', fontsize=10)
    ax.set_title('Class Distribution — Financial PhraseBank', fontsize=13)
    ax.set_ylabel('Count')
    ax.set_ylim(0, counts.max() * 1.18)
    plt.tight_layout()
    plt.show()


In [ ]:
# ── Load and explore ─────────────────────────────────────────────────────────
# Update path if needed
DATA_PATH = 'all-data.csv'

df = load_data(DATA_PATH)
df = preprocess_data(df)
explore_data(df)


## 3. Lexicon-Based Methods

### 3.1 — VADER

VADER (Valence Aware Dictionary and sEntiment Reasoner) uses a hand-crafted lexicon of
~7,500 tokens with valence scores. The compound score $\in [-1, 1]$ is mapped to:
- **positive** if compound $\geq 0.05$
- **negative** if compound $\leq -0.05$
- **neutral** otherwise

VADER was designed for social media; it tends to underperform on formal financial language.


In [ ]:
def classify_vader(compound: float) -> str:
    """Map VADER compound score to a sentiment label."""
    if compound >= 0.05:  return 'positive'
    if compound <= -0.05: return 'negative'
    return 'neutral'


def run_vader(df: pd.DataFrame) -> pd.DataFrame:
    """Apply VADER to each news headline and add result columns."""
    log.info("Running VADER sentiment analysis")
    sid = SentimentIntensityAnalyzer()
    df = df.copy()
    df['vader_compound']  = df['news'].apply(lambda x: sid.polarity_scores(x)['compound'])
    df['vader_sentiment'] = df['vader_compound'].apply(classify_vader)
    log.info("VADER analysis complete")
    return df


### 3.2 — TextBlob

In [ ]:
def classify_textblob(polarity: float) -> str:
    """Map TextBlob polarity score to a sentiment label."""
    if polarity > 0:  return 'positive'
    if polarity < 0:  return 'negative'
    return 'neutral'


def run_textblob(df: pd.DataFrame) -> pd.DataFrame:
    """Apply TextBlob polarity analysis."""
    log.info("Running TextBlob sentiment analysis")
    df = df.copy()
    df['textblob_polarity']  = df['news'].apply(lambda x: TextBlob(x).sentiment.polarity)
    df['textblob_sentiment'] = df['textblob_polarity'].apply(classify_textblob)
    log.info("TextBlob analysis complete")
    return df


## 4. Evaluation Utilities

In [ ]:
def evaluate(y_true, y_pred, model_name: str, verbose: bool = True) -> dict:
    """
    Compute accuracy, macro-F1, weighted-F1, and per-class F1.
    Prints a full classification report and returns a metrics dict.
    """
    acc    = accuracy_score(y_true, y_pred)
    f1_mac = f1_score(y_true, y_pred, average='macro',    labels=LABEL_ORDER, zero_division=0)
    f1_wt  = f1_score(y_true, y_pred, average='weighted', labels=LABEL_ORDER, zero_division=0)
    f1_cls = f1_score(y_true, y_pred, average=None,       labels=LABEL_ORDER, zero_division=0)

    if verbose:
        print(f"\n{'='*55}")
        print(f"  {model_name}")
        print(f"{'='*55}")
        print(f"  Accuracy:       {acc:.4f}")
        print(f"  F1 (macro):     {f1_mac:.4f}")
        print(f"  F1 (weighted):  {f1_wt:.4f}")
        print(f"\n{classification_report(y_true, y_pred, labels=LABEL_ORDER, zero_division=0)}")

    return {
        'model': model_name, 'accuracy': acc,
        'f1_macro': f1_mac, 'f1_weighted': f1_wt,
        'f1_negative': f1_cls[0], 'f1_neutral': f1_cls[1], 'f1_positive': f1_cls[2],
    }


def plot_confusion_matrix(y_true, y_pred, model_name: str) -> None:
    """Annotated heatmap of the confusion matrix."""
    cm = confusion_matrix(y_true, y_pred, labels=LABEL_ORDER)
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=LABEL_ORDER, yticklabels=LABEL_ORDER,
                linewidths=0.5, ax=ax)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
    ax.set_title(f'Confusion Matrix — {model_name}', fontsize=12)
    plt.tight_layout()
    plt.show()


def plot_model_comparison(results: list) -> None:
    """Bar chart comparing accuracy and F1 across all models."""
    res_df = pd.DataFrame(results).set_index('model')
    metrics = ['accuracy', 'f1_macro', 'f1_weighted', 'f1_negative', 'f1_neutral', 'f1_positive']
    res_df[metrics].plot(kind='bar', figsize=(13, 5), edgecolor='white', linewidth=0.8)
    plt.title('Model Comparison — All Metrics', fontsize=13)
    plt.ylabel('Score')
    plt.xticks(rotation=25, ha='right')
    plt.ylim(0, 1.0)
    plt.legend(loc='lower right', fontsize=9)
    plt.tight_layout()
    plt.show()


## 5. Machine Learning Classifiers

### 5.1 — TF-IDF + Sklearn Pipelines

Using `Pipeline(TfidfVectorizer → Classifier)` ensures that TF-IDF is fitted only on training
data in each CV fold — no leakage. Class imbalance (negative: 12.5% of data) is addressed via
`class_weight='balanced'`.


In [ ]:
def build_pipelines() -> dict:
    """
    Return a dict of name → (Pipeline, param_grid) ready for GridSearchCV.
    All classifiers use class_weight='balanced' to handle the neutral-heavy distribution.
    """
    tfidf_params = {
        'tfidf__max_features': [5000, 10000],
        'tfidf__ngram_range':  [(1, 1), (1, 2)],
        'tfidf__sublinear_tf': [True],
    }

    pipelines = {
        'Logistic Regression': (
            Pipeline([
                ('tfidf', TfidfVectorizer()),
                ('clf',   LogisticRegression(class_weight='balanced',
                                             max_iter=500,
                                             random_state=RANDOM_STATE))
            ]),
            {**tfidf_params, 'clf__C': [0.1, 1.0, 10.0]}
        ),
        'Linear SVM': (
            Pipeline([
                ('tfidf', TfidfVectorizer()),
                ('clf',   LinearSVC(class_weight='balanced',
                                    max_iter=2000,
                                    random_state=RANDOM_STATE))
            ]),
            {**tfidf_params, 'clf__C': [0.1, 1.0, 5.0]}
        ),
        'Random Forest': (
            Pipeline([
                ('tfidf', TfidfVectorizer()),
                ('clf',   RandomForestClassifier(class_weight='balanced',
                                                 n_jobs=-1,
                                                 random_state=RANDOM_STATE))
            ]),
            {**tfidf_params,
             'clf__n_estimators': [100, 200],
             'clf__max_depth':    [20, None]}
        ),
    }
    return pipelines


def train_models(df: pd.DataFrame) -> tuple:
    """
    Train all classifiers via stratified 5-fold GridSearchCV.
    Returns (best_models dict, results list, X_test, y_test).
    """
    X = df['news'].values
    y = df['sentiment'].values

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
    )
    log.info(f"Train: {len(X_train)} | Test: {len(X_test)}")

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    pipelines   = build_pipelines()
    best_models = {}
    results     = []

    for name, (pipe, params) in pipelines.items():
        log.info(f"Grid search: {name}")
        gs = GridSearchCV(pipe, params, cv=cv, scoring='f1_macro',
                          n_jobs=-1, verbose=0, refit=True)
        gs.fit(X_train, y_train)
        best_models[name] = gs.best_estimator_

        y_pred = gs.predict(X_test)
        metrics = evaluate(y_test, y_pred, name, verbose=True)
        results.append(metrics)
        plot_confusion_matrix(y_test, y_pred, name)
        log.info(f"{name} best params: {gs.best_params_}")

    return best_models, results, X_test, y_test


## 6. Full Pipeline Execution

In [ ]:
# ── Lexicon-based analysis ────────────────────────────────────────────────────
df = run_vader(df)
df = run_textblob(df)

vader_metrics   = evaluate(df['sentiment'], df['vader_sentiment'],   'VADER')
textblob_metrics = evaluate(df['sentiment'], df['textblob_sentiment'], 'TextBlob')

plot_confusion_matrix(df['sentiment'], df['vader_sentiment'],   'VADER')
plot_confusion_matrix(df['sentiment'], df['textblob_sentiment'], 'TextBlob')


In [ ]:
# ── ML classifiers ───────────────────────────────────────────────────────────
best_models, ml_results, X_test, y_test = train_models(df)

# ── Comparison across all models ──────────────────────────────────────────────
all_results = [vader_metrics, textblob_metrics] + ml_results
plot_model_comparison(all_results)

summary = pd.DataFrame(all_results).set_index('model')
print("\nFull Results Summary:")
print(summary[['accuracy', 'f1_macro', 'f1_weighted',
               'f1_negative', 'f1_neutral', 'f1_positive']].round(4).to_string())


## 7. Model Persistence & Inference

In [ ]:
# Save the best ML model (by macro-F1 on test set)
best_name = max(ml_results, key=lambda x: x['f1_macro'])['model']
best_pipe  = best_models[best_name]

joblib.dump(best_pipe, 'best_sentiment_model.pkl')
log.info(f"Saved best model ({best_name}) to best_sentiment_model.pkl")
print(f"Best model: {best_name}")


def predict_sentiment(texts: list, model_path: str = 'best_sentiment_model.pkl') -> pd.DataFrame:
    """
    Load the saved model and predict sentiment for a list of news strings.

    Returns a DataFrame with columns: news, sentiment, confidence (max class probability,
    only available for models with predict_proba).
    """
    model = joblib.load(model_path)
    preds = model.predict(texts)
    results = pd.DataFrame({'news': texts, 'sentiment': preds})

    if hasattr(model, 'predict_proba'):
        proba = model.predict_proba(texts)
        results['confidence'] = proba.max(axis=1)

    return results


# Example inference
sample_headlines = [
    "The company reported record-breaking quarterly profits, exceeding analyst expectations.",
    "Shares plummeted after the CEO announced unexpected restructuring charges.",
    "The board approved a dividend of $0.45 per share for the next quarter.",
    "Trading volume remained flat amid thin summer liquidity.",
]

preds = predict_sentiment(sample_headlines)
print("\nSample Predictions:")
print(preds.to_string(index=False))


## 8. Unit Tests

In [ ]:
class TestSentimentPipeline(unittest.TestCase):

    def setUp(self):
        self.df_sample = pd.DataFrame({
            'sentiment': ['positive', 'negative', 'neutral'],
            'news': [
                'The company posted strong earnings, beating all forecasts.',
                'Shares collapsed after the profit warning shocked markets.',
                'The board held its annual general meeting with no major announcements.',
            ]
        })

    def test_classify_vader_positive(self):
        self.assertEqual(classify_vader(0.5), 'positive')

    def test_classify_vader_negative(self):
        self.assertEqual(classify_vader(-0.5), 'negative')

    def test_classify_vader_neutral(self):
        self.assertEqual(classify_vader(0.0), 'neutral')

    def test_classify_textblob_positive(self):
        self.assertEqual(classify_textblob(0.3), 'positive')

    def test_classify_textblob_negative(self):
        self.assertEqual(classify_textblob(-0.3), 'negative')

    def test_classify_textblob_neutral(self):
        self.assertEqual(classify_textblob(0.0), 'neutral')

    def test_run_vader_columns(self):
        df_out = run_vader(self.df_sample)
        self.assertIn('vader_compound', df_out.columns)
        self.assertIn('vader_sentiment', df_out.columns)
        self.assertEqual(len(df_out), 3)
        self.assertTrue(df_out['vader_sentiment'].isin(LABEL_ORDER).all())

    def test_run_textblob_columns(self):
        df_out = run_textblob(self.df_sample)
        self.assertIn('textblob_polarity', df_out.columns)
        self.assertIn('textblob_sentiment', df_out.columns)
        self.assertEqual(len(df_out), 3)
        self.assertTrue(df_out['textblob_sentiment'].isin(LABEL_ORDER).all())

    def test_preprocess_removes_duplicates(self):
        df_dup = pd.concat([self.df_sample, self.df_sample])
        df_clean = preprocess_data(df_dup)
        self.assertEqual(len(df_clean), 3)

    def test_preprocess_removes_na(self):
        df_na = self.df_sample.copy()
        df_na.loc[0, 'news'] = None
        df_clean = preprocess_data(df_na)
        self.assertEqual(len(df_clean), 2)


# Run tests
suite = unittest.TestLoader().loadTestsFromTestCase(TestSentimentPipeline)
runner = unittest.TextTestRunner(verbosity=2)
runner.run(suite)
